# SkySense — Drone-View Object Detection with YOLOv8

Train a YOLOv8 model on the **VisDrone** dataset to detect **people, cars, motorbikes, and trucks** from aerial/drone imagery.

**Requirements:** Enable GPU runtime → `Runtime > Change runtime type > T4 GPU`

## 1. Configuration

Change these settings to scale the model or adjust training.

In [ ]:
# ============================================================
# CONFIGURATION — Edit these values to scale up/down
# ============================================================

# Model size: "yolov8n" (nano), "yolov8s" (small), "yolov8m" (medium),
#             "yolov8l" (large), "yolov8x" (extra-large)
# Bigger models are more accurate but need more GPU memory and training time.
# Free Colab T4 (16GB): use n/s/m. Colab Pro A100: use l/x.
MODEL_SIZE = "yolov8s"

# Training parameters
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16        # Reduce to 8 if you get OOM errors with larger models

# Target classes (mapped from VisDrone)
TARGET_CLASSES = ["person", "car", "motorbike", "truck"]

## 2. Setup & Install

In [ ]:
!pip install -q ultralytics

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

## 3. Download & Prepare VisDrone Dataset

VisDrone has 10 classes. We filter and remap to our 4 target classes:
- VisDrone `pedestrian` (0) + `people` (1) → **person** (0)
- VisDrone `car` (3) → **car** (1)
- VisDrone `motor` (5) → **motorbike** (2)
- VisDrone `truck` (9) → **truck** (3)

In [ ]:
import os
import yaml
from pathlib import Path
from ultralytics.data.utils import check_det_dataset

# Download VisDrone using Ultralytics' built-in downloader
VISDRONE_YAML = "VisDrone.yaml"
dataset_info = check_det_dataset(VISDRONE_YAML)
dataset_root = Path(dataset_info["path"])
print(f"Dataset downloaded to: {dataset_root}")

In [ ]:
from pathlib import Path
from tqdm import tqdm
import shutil
import yaml

# VisDrone class ID → our class ID mapping
# Only these classes are kept; all others are discarded
VISDRONE_REMAP = {
    0: 0,  # pedestrian → person
    1: 0,  # people     → person
    3: 1,  # car        → car
    5: 2,  # motor      → motorbike
    9: 3,  # truck      → truck
}

# Create filtered dataset directory
filtered_root = dataset_root.parent / "VisDrone_filtered"

for split in ["train", "val", "test"]:
    src_img_dir = dataset_root / "images" / split
    src_lbl_dir = dataset_root / "labels" / split
    dst_img_dir = filtered_root / "images" / split
    dst_lbl_dir = filtered_root / "labels" / split
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    if not src_lbl_dir.exists():
        print(f"Skipping {split} (not found)")
        continue

    label_files = list(src_lbl_dir.glob("*.txt"))
    kept, skipped = 0, 0

    for lbl_file in tqdm(label_files, desc=f"Processing {split}"):
        new_lines = []
        with open(lbl_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                if cls_id in VISDRONE_REMAP:
                    parts[0] = str(VISDRONE_REMAP[cls_id])
                    new_lines.append(" ".join(parts))

        # Only keep images that have at least one target annotation
        if new_lines:
            with open(dst_lbl_dir / lbl_file.name, "w") as f:
                f.write("\n".join(new_lines) + "\n")
            # Symlink image to save disk space
            img_name = lbl_file.stem
            for ext in [".jpg", ".jpeg", ".png"]:
                src_img = src_img_dir / (img_name + ext)
                if src_img.exists():
                    dst_img = dst_img_dir / (img_name + ext)
                    if not dst_img.exists():
                        shutil.copy2(src_img, dst_img)
                    break
            kept += 1
        else:
            skipped += 1

    print(f"{split}: kept {kept} images, skipped {skipped} (no target classes)")

# Write custom YAML config
custom_yaml_path = filtered_root / "dataset.yaml"
yaml_config = {
    "path": str(filtered_root),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": len(TARGET_CLASSES),
    "names": TARGET_CLASSES,
}
with open(custom_yaml_path, "w") as f:
    yaml.dump(yaml_config, f, default_flow_style=False)

print(f"\nCustom dataset config saved to: {custom_yaml_path}")
print(f"Classes: {TARGET_CLASSES}")

## 4. Train YOLOv8

Training starts from COCO-pretrained weights and fine-tunes on our filtered VisDrone data.  
To use a bigger model, change `MODEL_SIZE` in the Configuration cell above (e.g. `yolov8m`, `yolov8l`, `yolov8x`).

In [ ]:
from ultralytics import YOLO

# Load pretrained model
model = YOLO(f"{MODEL_SIZE}.pt")

# Train
results = model.train(
    data=str(custom_yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    project="skysense",
    name=f"{MODEL_SIZE}_visdrone",
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

print(f"\nTraining complete! Best weights: {results.save_dir}/weights/best.pt")

## 5. Evaluate Model

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Load best weights
best_weights = Path("skysense") / f"{MODEL_SIZE}_visdrone" / "weights" / "best.pt"
model = YOLO(str(best_weights))

# Run validation
metrics = model.val(data=str(custom_yaml_path), imgsz=IMG_SIZE, device=0)

print("\n" + "="*50)
print("EVALUATION RESULTS")
print("="*50)
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print()
for i, cls_name in enumerate(TARGET_CLASSES):
    print(f"  {cls_name:12s}  AP50: {metrics.box.ap50[i]:.4f}  AP50-95: {metrics.box.ap[i]:.4f}")

## 6. Inference & Visualization

Run the trained model on sample images and display detections.

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import random

# Get sample images from validation set
val_images = list((filtered_root / "images" / "val").glob("*.*"))
sample_images = random.sample(val_images, min(8, len(val_images)))

# Run inference
results = model.predict(
    source=sample_images,
    imgsz=IMG_SIZE,
    conf=0.25,
    device=0,
)

# Display results
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, (ax, result) in enumerate(zip(axes.flat, results)):
    annotated = result.plot()
    ax.imshow(annotated[..., ::-1])  # BGR → RGB
    ax.set_title(Path(result.path).name, fontsize=9)
    ax.axis("off")

# Hide unused subplots
for ax in axes.flat[len(results):]:
    ax.axis("off")

plt.suptitle(f"SkySense — {MODEL_SIZE} Detections on VisDrone", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Export Model

Export the trained model for deployment.

In [ ]:
from google.colab import files as colab_files

# Export to ONNX
onnx_path = model.export(format="onnx", imgsz=IMG_SIZE)
print(f"ONNX model exported to: {onnx_path}")

# Download best PyTorch weights
print(f"\nBest weights: {best_weights}")
print("Downloading weights...")
colab_files.download(str(best_weights))

---

## Scaling Guide

| Model | Size | mAP (COCO) | Speed (T4) | GPU RAM |
|-------|------|-----------|------------|----------|
| `yolov8n` | 3.2M | 37.3 | Fastest | ~4 GB |
| `yolov8s` | 11.2M | 44.9 | Fast | ~6 GB |
| `yolov8m` | 25.9M | 50.2 | Medium | ~10 GB |
| `yolov8l` | 43.7M | 52.9 | Slow | ~14 GB |
| `yolov8x` | 68.2M | 53.9 | Slowest | ~18 GB |

To scale up: change `MODEL_SIZE` in Cell 1 and reduce `BATCH_SIZE` if needed.